# Collating ALeRCE probabilities

One column per (classifier, class), filled for every object in a single vectorised pass.

The straightforward version of this — for each object, mask the probability table, then loop over
its classifiers and classes — costs one full scan of ~1M rows per object, so 100k objects is ~10^11
row comparisons. Building the whole matrix with `searchsorted` instead is one scan of the
probability table, total.


In [1]:
import time

import numpy as np
from astropy.table import Table, hstack


def _plain(table, column, dtype=None):
    """Column as a plain ndarray.

    FITS tables come back masked (which breaks np.unique(axis=)) and their string
    columns come back as bytes (which would name a column "clf1_b'SN'").
    """
    col = table[column]
    out = np.asarray(col.filled() if hasattr(col, "filled") else col)
    if out.dtype.kind == "S":
        out = np.char.decode(out, "utf-8")
    return out.astype(dtype) if dtype else out

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


# Load data

In [2]:
ALERCE = "/Users/leo/Projects/PHYS367/alerce"

diaObj = Table.read(f"{ALERCE}/alerce_lsstDiaObj_top100k_ndiasourcesGrt10_19Nov26.fits")
probs = Table.read(f"{ALERCE}/alerce_probs_forTop100k_ndiasourcesGrt10_19Nov26.fits")
taxonomy = Table.read(f"{ALERCE}/alerce_taxonomy.fits")

print(f"{len(diaObj):,} objects, {len(probs):,} probability rows, {len(taxonomy)} taxonomy entries")

100,000 objects, 995,226 probability rows, 69 taxonomy entries


### Check the two exports describe the same objects

`SELECT TOP n` with no `ORDER BY` returns an *arbitrary* subset, and two separate queries return
*different* arbitrary subsets — so an object export and a probability export fetched separately do
not necessarily cover the same oids. Measured on the files in `alerce/`:

| object file | probability file | objects covered |
|---|---|---|
| `top100k` | `probs_forTop100k` | 99.0% |
| `top100k_ndiasourcesGrt10` | `probs_forTop100k_ndiasourcesGrt10` | **69.2%** |
| `top200k` | `probs_forTop200k_cappedAt1M` | **0%** — disjoint |

The 200k pair shares no objects at all: that probability export hit the 1,000,000-row ceiling after
covering 153,335 oids, none of them in the object file. Worth re-exporting both sides from one
query before trusting any of this — there is a suggested ADQL at the bottom of the notebook.


In [3]:
def export_overlap(diaObj, probs):
    """What fraction of the object catalogue the probability export actually covers."""
    objects = np.unique(_plain(diaObj, "oid", "int64"))
    scored = np.unique(_plain(probs, "oid", "int64"))
    shared = np.intersect1d(objects, scored, assume_unique=True)
    print(f"{len(objects):,} objects, {len(scored):,} scored oids, {len(shared):,} shared "
          f"({100 * len(shared) / len(objects):.1f}% of the catalogue)")
    return shared


_ = export_overlap(diaObj, probs)

100,000 objects, 100,000 scored oids, 69,231 shared (69.2% of the catalogue)


# Build the probability matrix

`class_id` is only meaningful alongside `classifier_id` — class 0 is `SN` for classifier 1 but
`CEP` for classifier 9 — so the column key is the **pair**. The two index arrays are built once:

- **row** — `searchsorted` of each probability row's `oid` into the sorted object list
- **column** — `np.unique(..., return_inverse=True)` over the encoded `(classifier_id, class_id)` pair

then a single fancy-indexed assignment fills the matrix. Objects with no entry for a given
classifier keep `NaN`, which is what you want for "this classifier never scored this object" and
distinguishes it from a genuine probability of 0.


In [4]:
def probability_matrix(probs, taxonomy, oids=None, prefix="clf", dtype=np.float32):
    """Wide (n_objects x n_classifier_class) probability table.

    Returns (oids, Table) where each column is one (classifier, class) pair and each
    row is one object. Unscored combinations are NaN.

    One pass over `probs`: O(n_probs log n_objects), no Python-level loop over objects.
    """
    oid = _plain(probs, "oid", "int64")
    clf = _plain(probs, "classifier_id", "int64")
    cls = _plain(probs, "class_id", "int64")
    prob = _plain(probs, "probability", dtype)

    if oids is None:
        oids = np.unique(oid)
    else:
        oids = np.unique(np.asarray(oids, dtype="int64"))

    # Later classifier versions overwrite earlier ones for the same (oid, classifier, class).
    if "classifier_version" in probs.colnames:
        order = np.argsort(_plain(probs, "classifier_version", "int64"), kind="stable")
        oid, clf, cls, prob = oid[order], clf[order], cls[order], prob[order]

    row = np.searchsorted(oids, oid)
    in_range = row < len(oids)
    keep = np.zeros(len(oid), dtype=bool)
    keep[in_range] = oids[row[in_range]] == oid[in_range]

    key = clf * 1000 + cls                      # class_id is always < 1000
    unique_key, col = np.unique(key, return_inverse=True)

    matrix = np.full((len(oids), unique_key.size), np.nan, dtype=dtype)
    matrix[row[keep], col[keep]] = prob[keep]

    # Name each column from the taxonomy, which keys on (classifier_id, class_id) too.
    tax_key = _plain(taxonomy, "classifier_id", "int64") * 1000 + _plain(taxonomy, "class_id", "int64")
    tax_name = {int(k): str(n) for k, n in zip(tax_key, _plain(taxonomy, "class_name"))}
    names = [f"{prefix}{k // 1000}_{tax_name.get(int(k), f'class{k % 1000}')}" for k in unique_key]

    dropped = int((~keep).sum())
    if dropped:
        print(f"note: {dropped:,} probability rows refer to oids outside the requested list")

    return oids, Table(matrix, names=names)


t0 = time.perf_counter()
oids, wide = probability_matrix(probs, taxonomy)
print(f"{len(oids):,} objects x {len(wide.colnames)} columns in {time.perf_counter() - t0:.2f} s")
print(wide.colnames)
wide[:5]

100,000 objects x 12 columns in 0.16 s
['clf1_SN', 'clf1_AGN', 'clf1_VS', 'clf1_asteroid', 'clf1_bogus', 'clf3_SN', 'clf3_AGN', 'clf3_VS', 'clf3_asteroid', 'clf3_bogus', 'clf10_SNoffnuclear', 'clf10_notSNoffnuclear']


clf1_SN,clf1_AGN,clf1_VS,clf1_asteroid,clf1_bogus,clf3_SN,clf3_AGN,clf3_VS,clf3_asteroid,clf3_bogus,clf10_SNoffnuclear,clf10_notSNoffnuclear
float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32
nan,nan,nan,nan,nan,0.0077499696,0.94150287,0.03975922,0.0016461791,0.009341778,nan,nan
nan,nan,nan,nan,nan,0.0123322,0.9477229,0.033073224,0.001535393,0.0053363508,nan,nan
nan,nan,nan,nan,nan,0.0024161139,0.9559722,0.022764968,0.0007624367,0.018084453,nan,nan
nan,nan,nan,nan,nan,0.0055033555,0.83559114,0.06549456,0.0031364113,0.090274595,nan,nan
nan,nan,nan,nan,nan,0.007934837,0.9201298,0.015538625,0.00044194455,0.055954874,nan,nan


# Attach to the object catalogue

`searchsorted` again, this time to place each object's row of the matrix next to its catalogue row.
Objects absent from the probability table get a row of `NaN`.


In [5]:
def attach_probabilities(diaObj, oids, wide, keep_columns=("oid", "ra", "dec")):
    """Left-join the probability matrix onto the object catalogue, on oid."""
    target = _plain(diaObj, "oid", "int64")

    row = np.searchsorted(oids, target)
    in_range = row < len(oids)
    found = np.zeros(len(target), dtype=bool)
    found[in_range] = oids[row[in_range]] == target[in_range]
    row[~found] = 0                                   # placeholder, blanked below

    aligned = wide[row]
    for name in aligned.colnames:
        aligned[name][~found] = np.nan

    combined = hstack([diaObj[list(keep_columns)], aligned])
    print(f"{found.sum():,} of {len(target):,} objects matched "
          f"({100 * found.mean():.1f}%); {(~found).sum():,} left as NaN")
    return combined


t0 = time.perf_counter()
combined = attach_probabilities(diaObj, oids, wide)
print(f"joined in {time.perf_counter() - t0:.2f} s")
combined[:5]

69,231 of 100,000 objects matched (69.2%); 30,769 left as NaN
joined in 0.05 s


oid,ra,dec,clf1_SN,clf1_AGN,clf1_VS,clf1_asteroid,clf1_bogus,clf3_SN,clf3_AGN,clf3_VS,clf3_asteroid,clf3_bogus,clf10_SNoffnuclear,clf10_notSNoffnuclear
,deg,deg,,,,,,,,,,,,
int64,float64,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32
170019696011640884,53.19469873858197,-28.74241923722313,0.08551471,0.7449249,0.03145027,0.0017653503,0.13634478,0.5835283,0.27947074,0.09340196,0.007968761,0.0356303,nan,nan
170019696257007742,61.58219759057267,-49.563630902132275,0.00095058116,0.07370395,0.45489916,0.00091423985,0.46953204,0.0060957954,0.24206299,0.17499185,0.0022285215,0.57462084,nan,nan
170019696272736280,61.43427864369337,-49.43356458895318,0.0004465202,0.017969584,0.34367886,0.0017116723,0.6361934,0.008760231,0.16541447,0.3678667,0.0018272494,0.4561313,nan,nan
170019696277454895,60.58977373072803,-49.886315568996444,0.00048909767,0.0047770897,0.005554899,0.9240948,0.06508414,0.004443313,0.0011356683,0.0012335146,0.97805583,0.015131641,nan,nan
170019696279027843,60.29458319459662,-49.55741844816567,0.0033793892,0.4610742,0.29817924,0.0016450646,0.23572214,0.0067164013,0.7328122,0.088727824,0.0022527925,0.16949075,nan,nan


# Top class per classifier

The same matrix answers "what did each classifier call this object" — `nanargmax` along the
columns belonging to one classifier, with no second pass over the probability table.


In [6]:
def top_class(wide, prefix="clf"):
    """Best-scoring class and its probability, per classifier."""
    groups = {}
    for name in wide.colnames:
        groups.setdefault(name.split("_", 1)[0], []).append(name)

    out = Table()
    for classifier, columns in groups.items():
        block = np.column_stack([wide[c] for c in columns])
        scored = ~np.all(np.isnan(block), axis=1)
        best = np.full(len(block), -1)
        best[scored] = np.nanargmax(block[scored], axis=1)

        labels = np.array([c.split("_", 1)[1] for c in columns])
        out[f"{classifier}_class"] = np.where(scored, labels[best], "")
        out[f"{classifier}_prob"] = np.where(scored, np.nanmax(np.where(np.isnan(block), -np.inf, block), axis=1), np.nan)
    return out


t0 = time.perf_counter()
best = top_class(wide)
print(f"{time.perf_counter() - t0:.2f} s")
for column in best.colnames:
    if column.endswith("_class"):
        values, counts = np.unique(best[column], return_counts=True)
        print(f"{column}: " + ", ".join(f"{v or '(unscored)'}={c:,}" for v, c in zip(values, counts)))

0.05 s
clf1_class: (unscored)=1,046, AGN=16,056, SN=535, VS=28,530, asteroid=134, bogus=53,699
clf3_class: AGN=28,471, SN=1,159, VS=14,077, asteroid=281, bogus=56,012
clf10_class: (unscored)=99,772, SNoffnuclear=58, notSNoffnuclear=170


# Check it against the row-at-a-time version

The original `getProbabilities()` on a handful of objects, compared against the matrix, so the
speed-up is verifiably the same answer.


In [ ]:
def getProbabilities(i):
    """The original per-object version, kept as a reference implementation."""
    evtClass = probs[probs["oid"] == diaObj["oid"][i]]
    evtProbs = dict()
    for classifier in np.unique(evtClass["classifier_id"]):
        evtProbs[classifier] = dict()
        for id in evtClass["class_id"][evtClass["classifier_id"] == classifier]:
            name = taxonomy[(taxonomy["class_id"] == id) & (taxonomy["classifier_id"] == classifier)]["class_name"]
            if len(name) == 1:
                evtProbs[classifier][name[0]] = evtClass[(evtClass["classifier_id"] == classifier)
                                                         & (evtClass["class_id"] == id)]["probability"][0]
    return evtProbs


rng = np.random.default_rng(0)
sample = rng.choice(len(diaObj), size=25, replace=False)

t0 = time.perf_counter()
mismatches = 0
for i in sample:
    reference = getProbabilities(int(i))
    for classifier, classes in reference.items():
        for class_name, p in classes.items():
            column = f"clf{int(classifier)}_{class_name}"
            if not np.isclose(combined[column][int(i)], p, rtol=0, atol=1e-7):
                mismatches += 1
                print(f"  mismatch: row {i} {column}: {combined[column][int(i)]} != {p}")
loop_time = time.perf_counter() - t0

print(f"{mismatches} mismatches over {len(sample)} objects")
print(f"row-at-a-time: {loop_time:.1f} s for {len(sample)} objects "
      f"-> {loop_time / len(sample) * len(diaObj) / 60:.0f} min for all {len(diaObj):,}")

# Save

Plain float columns, so this writes to FITS without complaint — unlike a column of dicts, which
`Table.write` rejects with *"unsupported object types"*.


In [7]:
combined.write(f"{ALERCE}/alerce_lsstDiaObj_probabilities_wide.fits", overwrite=True)
print(f"{len(combined):,} rows x {len(combined.colnames)} columns")

100,000 rows x 15 columns


# Re-exporting the two sides consistently

To guarantee both files cover the same objects, pick the objects with a deterministic `ORDER BY`,
then fetch the probabilities for *that* oid range rather than a separate `TOP n`:

```sql
-- objects: 100k, deterministic
SELECT TOP 100000 o.oid, o.meanra, o.meandec, o.lastmjd, o.n_det
FROM alerce_tap.object AS o
WHERE o.sid = 1 AND o.n_det > 10
ORDER BY o.oid

-- probabilities for exactly those objects: same WHERE, joined server-side
SELECT p.oid, p.classifier_id, p.classifier_version, p.class_id, p.probability, p.ranking
FROM alerce_tap.probability AS p
JOIN (SELECT TOP 100000 oid FROM alerce_tap.object
      WHERE sid = 1 AND n_det > 10 ORDER BY oid) AS o ON o.oid = p.oid
WHERE p.sid = 1
ORDER BY p.oid
```

Mind the ceiling: 100k objects x ~10 class rows each is 1,000,000 — exactly the per-query limit, so
that second query will truncate. Either drop to ~90k objects, pin `classifier_id` to the one you
need (classifier 3 is the current Rubin stamp classifier, 5 classes, so 500k rows), or page it by
oid as `recent_lsst_objects` does in `queryALeRCE_TAP.ipynb`.
